In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import sqlite3
import pandas as pd

SCHEMA = """
CREATE TABLE images (
    id     INTEGER PRIMARY KEY AUTOINCREMENT,
    path   TEXT NOT NULL UNIQUE,
    label  TEXT NOT NULL CHECK (label IN ('circle','triangle','square','pentagon','hexagon')),
    source TEXT NOT NULL,
    split  TEXT NOT NULL CHECK (split IN ('train','test'))
);
"""
with open("src/schema.sql", "w") as f:
    f.write(SCHEMA)

conn = sqlite3.connect("data/shapes.db")
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS images")
cur.executescript(SCHEMA)

# load the dataset table into the database
data = pd.read_csv("data/processed/dataset.csv")
data.to_sql("images", conn, if_exists="append", index=False)
conn.commit()

print(pd.read_sql("SELECT COUNT(*) AS n FROM images", conn), len(data))

      n
0  4150 4150


In [3]:
# CREATE
cur.execute("INSERT INTO images (path, label, source, split) VALUES (?,?,?,?)",
            ("demo/fake.png", "circle", "generated", "train"))
conn.commit()
new_id = cur.lastrowid

# READ
print(pd.read_sql("SELECT * FROM images WHERE id = ?", conn, params=(new_id,)))

# UPDATE (fix a wrong label)
cur.execute("UPDATE images SET label = 'square' WHERE id = ?", (new_id,))
conn.commit()
print(pd.read_sql("SELECT * FROM images WHERE id = ?", conn, params=(new_id,)))

# DELETE
cur.execute("DELETE FROM images WHERE id = ?", (new_id,))
conn.commit()
print(pd.read_sql("SELECT COUNT(*) AS n FROM images", conn))

     id           path   label     source  split
0  4152  demo/fake.png  circle  generated  train
     id           path   label     source  split
0  4152  demo/fake.png  square  generated  train
      n
0  4150


In [5]:
# class balance per source and split
q = """
SELECT source, split, label, COUNT(*) AS n
FROM images
GROUP BY source, split, label
ORDER BY source, split, label
"""
pd.read_sql(q, conn).pivot_table(index=["source", "split"], columns="label", values="n")

# training images of one class from one source
pd.read_sql("""SELECT id, path FROM images
               WHERE split='train' AND source='kaggle_2d' AND label='hexagon'
               LIMIT 5""", conn)
conn.close()